In [1]:
import pandas as pd

MISSING = ["NA", "PS", "NULL", ""]
fos = pd.read_csv("../data/raw/Most-Recent-Cohorts-Field-of-Study.csv", dtype=str, keep_default_na=False)
scope_ids = pd.read_csv("../data/scope_unitids.csv", dtype=str)

print(fos.shape)
fos.iloc[:5, :11]

(227980, 178)


,UNITID,OPEID6,INSTNM,CONTROL,MAIN,CIPCODE,CIPDESC,CREDLEV,CREDDESC,IPEDSCOUNT1,IPEDSCOUNT2
0,100654,001002,Alabama A & M University,Public,1,0109,Animal Sciences.,3,Bachelor's Degree,9,18
1,100654,001002,Alabama A & M University,Public,1,0110,Food Science and Technology.,3,Bachelor's Degree,7,8
2,100654,001002,Alabama A & M University,Public,1,0110,Food Science and Technology.,5,Master's Degree,7,6
3,100654,001002,Alabama A & M University,Public,1,0110,Food Science and Technology.,6,Doctoral Degree,3,2
4,100654,001002,Alabama A & M University,Public,1,0199,Agricultural/Animal/Plant/Veterinary Science a...,3,Bachelor's Degree,1,2


In [2]:
key = ["UNITID", "CIPCODE", "CREDLEV"]
print(fos.duplicated(key).sum())
print((fos["UNITID"] == "NA").sum())
print(fos[fos["UNITID"] != "NA"].duplicated(key).sum())

6033
7020
0


In [3]:
fos[["CREDLEV", "CREDDESC"]].value_counts().sort_index()

CREDLEV  CREDDESC                                                             
1        Undergraduate Certificate or Diploma                                     43465
2        Associate's Degree                                                       42373
3        Bachelor's Degree                                                        71664
4        Post-baccalaureate Certificate                                            1249
5        Master's Degree                                                          38621
6        Doctoral Degree                                                          12637
7        First Professional Degree                                                 2350
8        Graduate/Professional Certificate                                        13921
99       Non-Credential Program (Preparatory Coursework/Teacher Certification)     1700
Name: count, dtype: int64

In [4]:
nursing = fos[(fos["CIPCODE"] == "5138") & (fos["CREDLEV"].isin(["2", "3"]))].copy()
print("Nursing rows:", len(nursing))
print(nursing["CIPDESC"].unique())

awards = (pd.to_numeric(nursing["IPEDSCOUNT1"], errors="coerce").fillna(0)
          + pd.to_numeric(nursing["IPEDSCOUNT2"], errors="coerce").fillna(0))
nursing = nursing[awards > 0]
print("With recent awards:", len(nursing))

merged = nursing.merge(scope_ids, on="UNITID", how="inner")
print("In our scope:", merged["UNITID"].nunique(), "colleges")
print(merged["CREDLEV"].value_counts())

Nursing rows: 2542
<ArrowStringArray>
['Registered Nursing, Nursing Administration, Nursing Research and Clinical Nursing.']
Length: 1, dtype: str
With recent awards: 2281
In our scope: 1820 colleges
CREDLEV
3    1042
2     957
Name: count, dtype: int64


In [5]:
cols = ["UNITID", "INSTNM", "CITY", "STABBR", "CONTROL", "LOCALE", "OPENADMP",
        "ADM_RATE", "ADMCON7", "COSTT4_A", "TUITIONFEE_IN", "NPT41_PUB", "NPT41_PRIV"]
inst = pd.read_csv("../data/raw/Most-Recent-Cohorts-Institution.csv", dtype=str,
                   keep_default_na=False, usecols=cols)

names = ["CUNY Bronx Community College", "CUNY Lehman College", "University of Tulsa"]
amina = inst[inst["INSTNM"].isin(names)]
amina.T

,1977,1987,2533
UNITID,190530,190637,207971
INSTNM,CUNY Bronx Community College,CUNY Lehman College,University of Tulsa
CITY,Bronx,Bronx,Tulsa
STABBR,NY,NY,OK
CONTROL,1,1,2
LOCALE,11,11,11
ADM_RATE,NA,0.5658,0.6151
NPT41_PUB,3667,1589,NA
NPT41_PRIV,NA,NA,5578
COSTT4_A,12677,14121,64459


In [6]:
merged[merged["UNITID"].isin(amina["UNITID"])][["INSTNM", "CREDLEV", "IPEDSCOUNT1", "IPEDSCOUNT2"]]

,INSTNM,CREDLEV,IPEDSCOUNT1,IPEDSCOUNT2
1031,CUNY Bronx Community College,2,21,10
1039,CUNY Lehman College,3,167,145
1344,University of Tulsa,3,22,39


| 11 | CIP 5138 includes nursing administration and RN-to-BSN completion programs, not only entry-level nursing | Next step: "Check whether this program admits students without a nursing license" |
| 12 | 7,020 field-of-study rows have UNITID = NA (closed or unmatched schools) | Drop before merging; key (UNITID, CIPCODE, CREDLEV) is then unique |
| 13 | Award counts can drop sharply between years (Bronx CC: 21 → 10) | Treat as a signal to verify, not as a quality judgment |
| 14 | Public COSTT4_A uses in-state tuition; residency is not confirmed | Public colleges add a residency next step |